In [1]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 425.6/425.6 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 264.7/264.7 kB 16.9 MB/s eta 0:00:00


In [2]:

import pandas as pd
import numpy as np
import optuna
import lightgbm as lgb
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score
from sklearn.preprocessing import LabelEncoder
import warnings
warnings.filterwarnings('ignore')

def df_Data_Cleansing(df):
    cat_cols = ['diet_type', 'stress_level', 'sleep_quality', 'physical_activity_level', 'smoking_alcohol', 'gender']
    for col in cat_cols:
        if col in df.columns:
            df[col] = df[col].astype(str).str.strip().astype('category')

    #14
    df['short_sleep_flag'] = (df['sleep_duration'] < 6.0).astype(int)

    #15
    df["exercise_strength"] = df['calorie_expenditure'] / ((df['exercise_duration'] + 0.1) * df['bmi'])

    #16
    df['healthy_score'] = (
        df['sleep_duration'].between(7, 8).astype(int) +
        (df['bmi'] <= 19).astype(int) +
        (df['exercise_duration'] >= 40).astype(int) +
        (df['stress_level'] == 'low').astype(int) +
        (df['sleep_quality'] == 'good').astype(int) +
        (df['physical_activity_level'] == 'active').astype(int)
    )

    #17
    df['unhealthy_score'] = (
        df['sleep_duration'].between(5, 6).astype(int) +
        (df['bmi'] >= 27).astype(int) +
        (df['calorie_expenditure'] <= 1400).astype(int) +
        (df['step_count'] <= 5000).astype(int) +
        (df['exercise_duration'] <= 30).astype(int) +
        (df['stress_level'] == 'high').astype(int) +
        (df['sleep_quality'] == 'poor').astype(int) +
        df['physical_activity_level'].isin(['sedentary', 'moderate']).astype(int) +
        (df['smoking_alcohol'] == 'yes').astype(int)
    )

    #18
    stress_mapping = {'low': 1, 'medium': 2, 'high': 3}
    df['stress_level_num'] = df['stress_level'].map(stress_mapping).astype(float)
    df['stress_per_sleep'] = df['stress_level_num'] / df['sleep_duration']

    # 不要な特徴量をまとめて削除
    drop_cols = [
        'stress_level_num',  # 計算用の一時データ（重複）
        'diet_type',
        'gender',
        'smoking_alcohol',
    ]
    df = df.drop(columns=drop_cols, errors='ignore')

    return df


# データの読み込み
df = df_Data_Cleansing(pd.read_csv('/content/train.csv', encoding="utf-8"))

X = df.drop(['id', 'health_condition'], axis=1)
y = df['health_condition']

# stress_levelのラベルエンコーディング（全体に適用して問題なし）
le_stress = LabelEncoder()
X['stress_level'] = le_stress.fit_transform(X['stress_level'])

# 欠損値補完用の数値カラムリストを取得
NUMS = X.select_dtypes(include="float").columns.tolist() + X.select_dtypes(include="int").columns.tolist()

le = LabelEncoder()
y_encoded = le.fit_transform(y)

# Optunaの目的関数
def objective(trial):
    params = {
        'objective': 'multiclass',
        'num_class': 3,
        'metric': 'multi_error',
        'verbosity': -1,
        'class_weight': 'balanced',
        'random_state': 42,
        'learning_rate': trial.suggest_float('learning_rate', 0.005, 0.05, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 10, 100),
        'max_depth': trial.suggest_int('max_depth', 3, 10),
        'min_child_samples': trial.suggest_int('min_child_samples', 10, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0)
    }

    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
    scores = []

    for train_idx, valid_idx in cv.split(X, y_encoded):
        # copy()を使用し、元のデータフレームへの影響を防ぐ
        X_train, X_valid = X.iloc[train_idx].copy(), X.iloc[valid_idx].copy()
        y_train, y_valid = y_encoded[train_idx], y_encoded[valid_idx]

        # === データリークを防ぐ欠損値補完 ===
        # 分割後のX_trainのみを使って中央値を計算
        train_medians = X_train[NUMS].median()
        X_train[NUMS] = X_train[NUMS].fillna(train_medians)
        X_valid[NUMS] = X_valid[NUMS].fillna(train_medians)
        # ====================================

        model = lgb.LGBMClassifier(**params, n_estimators=1000)

        # Early Stoppingを設定して過学習を防ぐ
        model.fit(
            X_train, y_train,
            eval_set=[(X_valid, y_valid)],
            callbacks=[lgb.early_stopping(stopping_rounds=50, verbose=False)]
        )

        preds = model.predict(X_valid)
        score = balanced_accuracy_score(y_valid, preds)
        scores.append(score)

    return np.mean(scores)

print("Optunaによるパラメータ探索を開始します...")
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=30) # まずは30回お試しで探索

print("=========================")
print(f"ベストスコア (Balanced Accuracy): {study.best_value:.5f}")
print("ベストパラメータ:")
for key, value in study.best_params.items():
    print(f"    {key}: {value}")
print("=========================")

[I 2026-07-21 10:48:45,303] A new study created in memory with name: no-name-491642c5-be61-4bd2-a985-173baafc1093


Optunaによるパラメータ探索を開始します...


[I 2026-07-21 10:50:55,504] Trial 0 finished with value: 0.9486541554175437 and parameters: {'learning_rate': 0.013578512419982092, 'num_leaves': 92, 'max_depth': 9, 'min_child_samples': 36, 'subsample': 0.505217415288255, 'colsample_bytree': 0.7581737202993579}. Best is trial 0 with value: 0.9486541554175437.
[I 2026-07-21 10:52:28,577] Trial 1 finished with value: 0.933294285621049 and parameters: {'learning_rate': 0.01879846770608017, 'num_leaves': 29, 'max_depth': 4, 'min_child_samples': 19, 'subsample': 0.9462234755266052, 'colsample_bytree': 0.8655699254352003}. Best is trial 0 with value: 0.9486541554175437.
[I 2026-07-21 10:55:44,814] Trial 2 finished with value: 0.948342589036376 and parameters: {'learning_rate': 0.005493765787874933, 'num_leaves': 87, 'max_depth': 8, 'min_child_samples': 67, 'subsample': 0.9516953924223616, 'colsample_bytree': 0.5549301957165631}. Best is trial 0 with value: 0.9486541554175437.
[I 2026-07-21 10:58:49,456] Trial 3 finished with value: 0.948553

ベストスコア (Balanced Accuracy): 0.94865
ベストパラメータ:
    learning_rate: 0.013578512419982092
    num_leaves: 92
    max_depth: 9
    min_child_samples: 36
    subsample: 0.505217415288255
    colsample_bytree: 0.7581737202993579
